In [8]:
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# Setting global visualization themes
sns.set_theme(style="whitegrid")
print("Environment successfully configured!")

Environment successfully configured!


Part 1: Feature Engineering — Computing RFM Metrics

In [9]:
# Setting random seed for reproducibility
import pandas as pd
import numpy as np
import datetime as dt

np.random.seed(42)
n_transactions = 15000
start_date = dt.datetime(2025, 1, 1)

# Generating synthetic transaction attributes
customer_ids = [f"CUST_{np.random.randint(1000, 2000)}" for _ in range(n_transactions)]
transaction_ids = [f"TXN_{str(i).zfill(6)}" for i in range(1, n_transactions + 1)]
random_days = np.random.randint(0, 365, size=n_transactions)
transaction_dates = [start_date + dt.timedelta(days=int(d)) for d in random_days]
order_values = np.round(np.random.exponential(scale=75, size=n_transactions) + 10, 2)

# Creating DataFrame (fixed indentation with standard spaces)
raw_df = pd.DataFrame({
    'TransactionID': transaction_ids,
    'CustomerID': customer_ids,
    'TransactionDate': transaction_dates,
    'OrderValue': order_values
})

# Save raw logs and inspect structure
raw_df.to_csv('raw_customer_transactions.csv', index=False)
print(f"Dataset successfully created with {len(raw_df)} transaction logs across {raw_df['CustomerID'].nunique()} unique customers!")
display(raw_df.head())

Dataset successfully created with 15000 transaction logs across 1000 unique customers!


,TransactionID,CustomerID,TransactionDate,OrderValue
0,TXN_000001,CUST_1102,2025-05-11,53.44
1,TXN_000002,CUST_1435,2025-03-08,318.28
2,TXN_000003,CUST_1860,2025-08-16,73.14
3,TXN_000004,CUST_1270,2025-06-12,27.00
4,TXN_000005,CUST_1106,2025-01-15,35.98


In [10]:
# Define snapshot date (1 day after the latest transaction date in the dataset)
snapshot_date = raw_df['TransactionDate'].max() + dt.timedelta(days=1)

# Group by CustomerID and compute RFM metrics
rfm_df = raw_df.groupby('CustomerID').agg({
'TransactionDate': lambda x: (snapshot_date - x.max()).days, # Recency
'TransactionID': 'count', # Frequency
'OrderValue': 'sum' # Monetary
}).reset_index()

# Rename columns to standard RFM nomenclature
rfm_df.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print(f"RFM aggregated for {len(rfm_df)} customers.")

RFM aggregated for 1000 customers.


In [11]:
rfm_df.head()

,CustomerID,Recency,Frequency,Monetary
0,CUST_1000,15,19,1586.42
1,CUST_1001,39,15,1061.24
2,CUST_1002,10,12,1240.64
3,CUST_1003,40,10,1003.76
4,CUST_1004,30,10,1058.63


Part 2: Feature Scaling & K-Means Clustering

In [12]:
# numerical features for clustering
rfm_features = rfm_df[['Recency', 'Frequency', 'Monetary']]

# Initializing and fitting StandardScaler
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

# Convert scaled array back to DataFrame for verification
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency_Scaled', 'Frequency_Scaled', 'Monetary_Scaled'])
rfm_scaled_df.head()

,Recency_Scaled,Frequency_Scaled,Monetary_Scaled
0,-0.404689,1.004151,0.700345
1,0.604721,0.000000,-0.504873
2,-0.614983,-0.753113,-0.093174
3,0.646779,-1.255188,-0.636782
4,0.226192,-1.255188,-0.510862


In [13]:
# Train K-Means model
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_df['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Inspect mean values for each cluster to evaluate performance
cluster_summary = rfm_df.groupby('Cluster').agg({
'Recency': 'mean',
'Frequency': 'mean',
'Monetary': ['mean', 'count']
}).round(2)

print("Cluster Summary (Mean Values):")
print(cluster_summary)

Cluster Summary (Mean Values):
        Recency Frequency Monetary      
           mean      mean     mean count
Cluster                                 
0         17.77     16.01  1356.02   430
1         20.62     11.42   885.01   295
2         78.38     11.85  1015.82   104
3         16.06     20.56  1938.18   171


In [14]:
# Define persona mapping based on cluster characteristics
# (Low Recency + High Frequency + High Monetary = Champions)
def assign_persona(row):
  if row['Cluster'] == 0:
    return 'At-Risk / Lost'
  elif row['Cluster'] == 1:
    return 'Champions'
  elif row['Cluster'] == 2:
    return 'Loyal Customers'
  else:
    return 'Recent / New Visitors'

  rfm_df['Customer_Persona'] = rfm_df.apply(assign_persona, axis=1)

# Export the labeled dataset for Tableau ingestion
rfm_df.to_csv('rfm_segmented_customers.csv', index=False)
print("Segmented dataset exported successfully as 'rfm_segmented_customers.csv'!")
rfm_df.head()

Segmented dataset exported successfully as 'rfm_segmented_customers.csv'!


,CustomerID,Recency,Frequency,Monetary,Cluster
0,CUST_1000,15,19,1586.42,3
1,CUST_1001,39,15,1061.24,0
2,CUST_1002,10,12,1240.64,1
3,CUST_1003,40,10,1003.76,1
4,CUST_1004,30,10,1058.63,1
